<a href="https://colab.research.google.com/github/MatthewTsan/RAG-Toy/blob/start/Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install sentence-transformers faiss-cpu beautifulsoup4 html2text tqdm


In [3]:
# === pick one of: "dataset1", "dataset2", "dataset3" ===
DATASET = "dataset1"  # <- change here

if DATASET == "dataset1":
    with open("dataset1.jsonl","w") as f:
        f.write('{"id":"solar_system","url":"https://simple.wikipedia.org/wiki/Solar_System","title":"Solar System","text":""}\n')
elif DATASET == "dataset2":
    with open("dataset2.jsonl","w") as f:
        f.write('{"id":"sun","url":"https://simple.wikipedia.org/wiki/Sun","title":"Sun","text":""}\n')
        f.write('{"id":"earth","url":"https://simple.wikipedia.org/wiki/Earth","title":"Earth","text":""}\n')
        f.write('{"id":"moon","url":"https://simple.wikipedia.org/wiki/Moon","title":"Moon","text":""}\n')
        f.write('{"id":"mars","url":"https://simple.wikipedia.org/wiki/Mars","title":"Mars","text":""}\n')
        f.write('{"id":"jupiter","url":"https://simple.wikipedia.org/wiki/Jupiter","title":"Jupiter","text":""}\n')
elif DATASET == "dataset3":
    with open("dataset3.jsonl","w") as f:
        f.write('{"id":"apollo11","url":"https://en.wikipedia.org/wiki/Apollo_11","title":"Apollo 11","text":""}\n')
        f.write('{"id":"armstrong","url":"https://en.wikipedia.org/wiki/Neil_Armstrong","title":"Neil Armstrong","text":""}\n')
        f.write('{"id":"aldrin","url":"https://en.wikipedia.org/wiki/Buzz_Aldrin","title":"Buzz Aldrin","text":""}\n')
        f.write('{"id":"saturnv","url":"https://en.wikipedia.org/wiki/Saturn_V","title":"Saturn V","text":""}\n')


In [4]:
import json, re, os, time, hashlib, requests
from bs4 import BeautifulSoup
import html2text
from tqdm import tqdm

def read_jsonl(path):
    docs = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                docs.append(json.loads(line))
    return docs

def fetch_url(url, retries=3, sleep=1.5):
    headers = {"User-Agent": "Colab-RAG/1.0 (+https://colab.google.com/)"}
    for i in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=20)
            r.raise_for_status()
            return r.text
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(sleep)

def html_to_markdown_text(html):
    # Remove navbars/footers/tables/refs roughly via BS4 first
    soup = BeautifulSoup(html, "html.parser")

    # Drop scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Wikipedia: remove tables, infoboxes, reference lists for simplicity
    for tag in soup.find_all(["table", "sup", "style", "img"]):
        tag.decompose()

    # Convert to markdown-like plain text
    h = html2text.HTML2Text()
    h.ignore_links = True
    h.ignore_images = True
    h.ignore_emphasis = False
    h.body_width = 0
    text = h.handle(str(soup))

    # Clean brackets like [1], [2], extra whitespace
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def chunk_text(text, max_chars=1200, overlap=200):
    """
    Robust char-based chunker that:
    - prefers breaking at a newline before `end`,
    - always advances `start` (no infinite loops),
    - enforces overlap but never moves backwards.
    """
    text = (text or "").strip()
    n = len(text)
    if n == 0:
        return []

    chunks = []
    start = 0

    # ensure sane params
    max_chars = max(1, int(max_chars))
    overlap = max(0, int(overlap))
    # effective step if we can't find a good split
    min_step = max(1, max_chars - overlap)

    while start < n:
        end = min(n, start + max_chars)

        # try to split at a newline strictly after `start`
        split = text.rfind("\n", start + 1, end)
        # if no newline found near the end window, just cut at `end`
        if split == -1 or (end - split) > int(max_chars * 0.6):
            split = end

        # slice & trim (avoid empty chunks)
        chunk = text[start:split].rstrip()
        if not chunk:
            # fallback: force progress by taking a minimal step
            split = min(n, start + min_step)
            chunk = text[start:split].rstrip()

        chunks.append(chunk)

        if split >= n:
            break

        # propose next start with overlap
        next_start = split - overlap

        # guarantee forward progress
        if next_start <= start:
            next_start = start + min_step

        start = min(n, next_start)

    return chunks



In [5]:
DATASET_PATH = {
    "dataset1": "dataset1.jsonl",
    "dataset2": "dataset2.jsonl",
    "dataset3": "dataset3.jsonl",
}[DATASET]

raw_docs = read_jsonl(DATASET_PATH)
len(raw_docs), raw_docs[:2]

(1,
 [{'id': 'solar_system',
   'url': 'https://simple.wikipedia.org/wiki/Solar_System',
   'title': 'Solar System',
   'text': ''}])

In [6]:
ingested = []  # [{doc_id, url, title, chunk_id, text}]
for d in tqdm(raw_docs, desc="Ingesting"):
    url = d["url"]
    html = fetch_url(url)
    # print("\nfetch html success")
    txt = html_to_markdown_text(html)
    # print("\nhtml to markdown success")
    chunks = chunk_text(txt, max_chars=1200, overlap=180)
    # print("chunk success")
    for i, ch in enumerate(chunks):
        ingested.append({
            "doc_id": d["id"],
            "url": url,
            "title": d.get("title", d["id"]),
            "chunk_id": i,
            "text": ch
        })

print(f"\nDocs: {len(raw_docs)}  |  Chunks: {len(ingested)}")


Ingesting: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


Docs: 1  |  Chunks: 95


In [7]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # ~384-dim

texts = [c["text"] for c in ingested]
embs = model.encode(texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True)

dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)          # cosine similarity because we normalized
index.add(embs)
print("Index size:", index.ntotal)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Index size: 95


In [8]:
os.makedirs("rag_index", exist_ok=True)
faiss.write_index(index, f"rag_index/{DATASET}.faiss")

with open(f"rag_index/{DATAET if 'DATAET' in globals() else DATASET}_chunks.jsonl","w") as f:
    for c in ingested:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

np.save(f"rag_index/{DATASET}_embeddings.npy", embs)
print("Saved to rag_index/")


Saved to rag_index/


In [9]:
# Reload (just to prove persistence)
index = faiss.read_index(f"rag_index/{DATASET}.faiss")
ing = [json.loads(l) for l in open(f"rag_index/{DATASET}_chunks.jsonl","r").read().splitlines()]

def search(query, k=5):
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q, k)
    results = []
    for score, idx in zip(D[0].tolist(), I[0].tolist()):
        if idx == -1:
            continue
        meta = ing[idx]
        results.append({"score": float(score), **meta})
    return results

# Try a few queries depending on dataset:
queries = {
    "dataset1": ["What is at the center of the Solar System?", "What is the Kuiper belt?"],
    "dataset2": ["Which planet has the Great Red Spot?", "What type of star is the Sun?"],
    "dataset3": ["Who was the command module pilot?", "What rocket launched Apollo 11?"],
}[DATASET]

for q in queries:
    print(f"\nQ: {q}")
    res = search(q, k=3)
    for r in res:
        print(f"- score={r['score']:.3f}  doc={r['doc_id']}  chunk={r['chunk_id']}  url={r['url']}")
        print(r['text'][:300].replace("\n"," ") + " ...")



Q: What is at the center of the Solar System?
- score=0.628  doc=solar_system  chunk=46  url=https://simple.wikipedia.org/wiki/Solar_System
used to be the Sun. Eventually, the Solar System could completely fall apart, with its planets and moons scattered across the galaxy. This slow breakup is a natural part of how gravity works over a long time, and it would be the end of the Solar System.   ## General Characteristics of the Solar Syst ...
- score=0.628  doc=solar_system  chunk=5  url=https://simple.wikipedia.org/wiki/Solar_System
d as PDF   * Page for printing  In other projects     * Wikimedia Commons   * Wikidata item  Appearance  move to sidebar hide  From Simple English Wikipedia, the free encyclopedia  Planets and dwarf planets of the Solar System. Compared with each other, the sizes are correct, but the distances are n ...
- score=0.589  doc=solar_system  chunk=47  url=https://simple.wikipedia.org/wiki/Solar_System
 the heliocentric system, which means “Sun-centered.” The Sola